**Agri-Sentry360** | Day 3: Farm Boundary Manager (farm_purifier) <br>
Author: Ranjit Saha <br>
**Objective:** Clean, validate, and secure farm boundaries for PostGIS and SAR Analysis.
 

In [6]:
# farm_boundary_manager.py

import os
import geopandas as gpd
from shapely.geometry import box
from shapely.validation import make_valid
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# --- HELPER: Professional Logging ---
def log_step(msg):
    """Internally works as a formatted print for clean terminal output."""
    print(f"|-- [LOG]: {msg}")

def purify_farms(input_path="outputs/dhaldabri_real_farms.geojson", output_dir="outputs"):
    print("=" * 65)
    print("   Agri-Sentry360: DHADLABRI FARM PURIFICATION PIPELINE")
    print("=" * 65)

    # Ensure output workspace exists
    os.makedirs(output_dir, exist_ok=True)

    # --- STEP 1: LOAD DATA ---
    if not os.path.exists(input_path):
        log_step(f"❌ Error: Input file {input_path} not found. Create the farm boundaries from 'geojson.io' ") 
        return None

    gdf = gpd.read_file(input_path, crs="EPSG:4326")
    log_step(f"Successfully loaded {len(gdf)} farm boundaries.")

    # Assigning farmer name's & farm id's 
    if 'farmer_name' not in gdf.columns:
        log_step("⚠️ 'farmer_name' missing! Assigning default IDs...")
        gdf['farmer_name'] = [f"Farmer_{i+1}" for i in range(len(gdf))]
        gdf['farm_id'] = [f"DHF{i+1:03d}" for i in range(len(gdf))]


    # --- STEP 2: GEOMETRY VALIDATION ("The Bowtie Fix") ---
    # Fixes self-intersections and topological errors
    gdf['geometry'] = gdf['geometry'].apply(
        lambda geom: make_valid(geom) if not geom.is_valid else geom
    )
    log_step("Geometry validation complete (All 'Bowtie' shapes untangled).")

    # --- STEP 3: ACCURATE SPATIAL MATH (UTM Zone 45N) ---
    # Convert to Meters (EPSG:32645) for precise West Bengal calculations
    gdf_utm = gdf.to_crs(epsg=32645)
    
    raw_area_m2 = gdf_utm.geometry.area
    gdf['area_bigha'] = (raw_area_m2 / 1337.805).round(4) # Bengal Bigha
    gdf['area_ha'] = (raw_area_m2 / 10_000).round(4)
    
    log_step(f"Area Calculation: Found {gdf['area_ha'].sum():.2f} Total Hectares.")
    log_step(f"Area Calculation: Found {gdf['area_bigha'].sum():.2f} Total Bighas.")


    # --- STEP 3.5: SANITY FILTERS (Tiny vs. Massive) ---
    # Define thresholds in Hectares (the math standard)
    MIN_AREA_HA = 0.05  
    MAX_AREA_HA = 25.0  

    # Find the troublemakers
    tiny_farms = gdf[gdf['area_ha'] < MIN_AREA_HA]
    massive_farms = gdf[gdf['area_ha'] > MAX_AREA_HA]

    if not tiny_farms.empty:
        # Link Bigha in the log so the user understands the scale
        # (0.05 Ha is roughly 0.37 Bigha)
        log_step(f"⚠️ Removing {len(tiny_farms)} 'Ghost' plots (Too small: < 0.4 Bigha).")
        gdf = gdf[gdf['area_ha'] >= MIN_AREA_HA]

    if not massive_farms.empty:
        # 25 Ha is roughly 187 Bigha
        log_step(f"❌ ALERT: Found {len(massive_farms)} massive plots (> 185 Bigha).")
        log_step("   -> These will be flagged for manual verification.")
        gdf['is_outlier'] = gdf['area_ha'] > MAX_AREA_HA
    else:
        log_step("✅ All farms within realistic Dhaldabri size range (0.4 to 185 Bigha).")

    
    
    # --- STEP 4: PRECISION LAYER (Inner Core) ---
    # Shrink inward by 3m to remove "Edge Noise" (roads, fences) for SAR data
    gdf_utm['geom_inner'] = gdf_utm.geometry.buffer(-3)
    gdf['geom_inner'] = gdf_utm['geom_inner'].to_crs(epsg=4326) # Back to Degrees
    log_step("3m 'Inner Core' generated for pure-pixel satellite sampling.")

    # --- STEP 5: PROJECT BOUNDING BOX (BBOX) ---
    # This is the "Satellite Window" for Day 4
    total_bounds = gdf.total_bounds
    log_step(f"BBox calculated: {total_bounds.tolist()} (Ready for GEE).")
    
    
    # --- STEP 6: PREPARE FOR POSTGIS ---
    # Database standard: rename 'geometry' column to 'geom'
    gdf = gdf.rename_geometry('geom')
    log_step("Geometry column standardized to 'geom' for PostGIS compatibility.")

    # --- STEP 7: SECURE STORAGE ---
    # Parquet is used because it can store multiple geometry columns securely
    output_parquet = os.path.join(output_dir, "dhaldabri_farms_secured.parquet")
    gdf.to_parquet(output_parquet)
    
    print("-" * 65)
    log_step(f"✅ SUCCESS: Purified data secured at {output_parquet}")
    print("=" * 65)

    return gdf, total_bounds

# --- MAIN EXECUTION ---
if __name__ == "__main__":
    # This triggers the purification process
    final_gdf, bbox = purify_farms()
    
    # Preview the data for the developer
    print("\nPreview of Purified Data (First 2 Rows):")
    print(final_gdf[ ['farm_id', 'farmer_name', 'area_bigha', 'area_ha']].head(2))


   Agri-Sentry360: DHADLABRI FARM PURIFICATION PIPELINE
|-- [LOG]: Successfully loaded 14 farm boundaries.
|-- [LOG]: ⚠️ 'farmer_name' missing! Assigning default IDs...
|-- [LOG]: Geometry validation complete (All 'Bowtie' shapes untangled).
|-- [LOG]: Area Calculation: Found 19.94 Total Hectares.
|-- [LOG]: Area Calculation: Found 149.08 Total Bighas.
|-- [LOG]: ⚠️ Removing 2 'Ghost' plots (Too small: < 0.4 Bigha).
|-- [LOG]: ✅ All farms within realistic Dhaldabri size range (0.4 to 185 Bigha).
|-- [LOG]: 3m 'Inner Core' generated for pure-pixel satellite sampling.
|-- [LOG]: BBox calculated: [89.72779859228876, 26.354099366642046, 89.73716359424549, 26.362171757502153] (Ready for GEE).
|-- [LOG]: Geometry column standardized to 'geom' for PostGIS compatibility.
-----------------------------------------------------------------
|-- [LOG]: ✅ SUCCESS: Purified data secured at outputs\dhaldabri_farms_secured.parquet

Preview of Purified Data (First 2 Rows):
  farm_id farmer_name  area_big

In [7]:
final_gdf

,geom,farmer_name,farm_id,area_bigha,area_ha,geom_inner
0,"POLYGON ((89.73357 26.3574, 89.73356 26.35715,...",Farmer_1,DHF001,1.1058,0.1479,"POLYGON ((89.7336 26.35738, 89.73374 26.3574, ..."
1,"POLYGON ((89.73377 26.35744, 89.73376 26.35675...",Farmer_2,DHF002,1.2349,0.1652,"POLYGON ((89.7338 26.35741, 89.73394 26.35742,..."
3,"POLYGON ((89.7341 26.35593, 89.73429 26.35595,...",Farmer_4,DHF004,0.5784,0.0774,"POLYGON ((89.73413 26.35596, 89.73409 26.35628..."
4,"POLYGON ((89.73395 26.35659, 89.73396 26.35631...",Farmer_5,DHF005,1.0193,0.1364,"POLYGON ((89.73398 26.35657, 89.73431 26.35661..."
5,"POLYGON ((89.73431 26.35691, 89.73435 26.35649...",Farmer_6,DHF006,0.9785,0.1309,"POLYGON ((89.73434 26.35688, 89.73456 26.3569,..."
6,"POLYGON ((89.7334 26.35623, 89.73342 26.35601,...",Farmer_7,DHF007,0.8139,0.1089,"POLYGON ((89.73343 26.35621, 89.73381 26.35627..."
7,"POLYGON ((89.73124 26.35464, 89.73129 26.35453...",Farmer_8,DHF008,2.1840,0.2922,"POLYGON ((89.73127 26.35463, 89.73161 26.35477..."
8,"POLYGON ((89.73099 26.35543, 89.73108 26.35527...",Farmer_9,DHF009,3.0212,0.4042,"POLYGON ((89.73102 26.35543, 89.73105 26.35552..."
9,"POLYGON ((89.72874 26.35704, 89.72802 26.35702...",Farmer_10,DHF010,29.3203,3.9225,"POLYGON ((89.72872 26.35701, 89.72893 26.35635..."
10,"POLYGON ((89.73253 26.3592, 89.73281 26.35924,...",Farmer_11,DHF011,106.0978,14.1938,"POLYGON ((89.73257 26.35923, 89.73273 26.36103..."


In [5]:
final_gdf.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [8]:
bbox

array([89.72779859, 26.35409937, 89.73716359, 26.36217176])

 
**Agri-Sentry360** | Day 3: PostGIS Vault Manager (db_helper)<br>
Author: Ranjit Saha

**Description:** Industrial-grade spatial persistence layer. 
            
**Features:** 
- 1. Idempotent Schema Creation (Fail-safe)
- 2. Spatial Integrity Constraints (Min/Max Area & Topology)
- 3. Row-Level Security (RLS) for Data Privacy
- 4. Dual-Geometry Indexing (GIST) for high-performance SAR sampling.
 

In [4]:
# db_helper.py | postgis_setup.py

import os
import pandas as pd
import geopandas as gpd
from sqlalchemy import create_engine, text
from geoalchemy2 import Geometry
from config import Config # <-- The Central Brain


# ARCHITECT'S NOTE: We use a connection pool to handle concurrent Day 4 satellite requests
engine = create_engine(Config.DB_URL, pool_size=10, max_overflow=20)

def ensure_vault_exists():
    """
    ARCHITECTURE: Implements an 'Idempotent' setup. 
    Checks for the table, constraints, and privacy policies without 
    destroying existing farmer data.
    """
    print(f"\n{'='*60}\n🏗️  PHASE 1: SECURING THE SPATIAL VAULT (RLS & CONSTRAINTS)\n{'='*60}")
  
    setup_sql = """
    CREATE EXTENSION IF NOT EXISTS postgis;
    
    -- 1. Create basic table if it doesn't exist
    CREATE TABLE IF NOT EXISTS farm_boundaries (
        farm_id         VARCHAR(20) PRIMARY KEY,
        farmer_name     VARCHAR(50) NOT NULL,
        owner_id        VARCHAR(50) DEFAULT 'postgres' NOT NULL, 
        crop_type       VARCHAR(50),
        area_hectares   DECIMAL(10, 4),
        area_bigha      DECIMAL(10, 4),
        geom            GEOMETRY(POLYGON, 4326),
        geom_inner      GEOMETRY(POLYGON, 4326),
        created_at      TIMESTAMP DEFAULT NOW()
    );

    -- 2. Migration: Add owner_id and Constraints to existing table if missing
    DO $$ 
    BEGIN 
        -- Add owner_id if missing
        IF NOT EXISTS (SELECT 1 FROM information_schema.columns 
                       WHERE table_name='farm_boundaries' AND column_name='owner_id') THEN
            ALTER TABLE farm_boundaries ADD COLUMN owner_id VARCHAR(50) DEFAULT 'postgres' NOT NULL;
        END IF;

        -- Add valid_geom constraint if missing
        IF NOT EXISTS (SELECT 1 FROM pg_constraint WHERE conname = 'enforce_valid_geom') THEN
            ALTER TABLE farm_boundaries ADD CONSTRAINT enforce_valid_geom CHECK (ST_IsValid(geom));
        END IF;

        -- Add area_range constraint if missing
        IF NOT EXISTS (SELECT 1 FROM pg_constraint WHERE conname = 'enforce_area_range') THEN
            ALTER TABLE farm_boundaries ADD CONSTRAINT enforce_area_range 
            CHECK (area_hectares > 0.01 AND area_hectares < 25.0);
        END IF;
    END $$;

    -- 3. Security & Privacy
    ALTER TABLE farm_boundaries ENABLE ROW LEVEL SECURITY;

    DO $$
    BEGIN
        IF NOT EXISTS (SELECT 1 FROM pg_policy WHERE polname = 'farm_privacy_policy') THEN
            CREATE POLICY farm_privacy_policy ON farm_boundaries
            FOR ALL USING (owner_id = current_user OR current_user = 'postgres');
        END IF;
    END $$;

    -- 4. Indexes
    CREATE INDEX IF NOT EXISTS idx_farm_geom ON farm_boundaries USING GIST(geom);
    CREATE INDEX IF NOT EXISTS idx_farm_inner ON farm_boundaries USING GIST(geom_inner);
    """


    

    with engine.connect() as conn:
        conn.execute(text(setup_sql))
        conn.commit()
    print("|-- [LOG]: Vault structure, RLS policies, and Spatial Indexes are active.")

     
    
def sync_farms(input_file="outputs/dhaldabri_farms_secured.parquet"):
    """
    ARCHITECTURE: Syncs new data with 'Owner' assignment. 
    Prevents duplicates while respecting the security schema.
    """
    print(f"\n{'='*60}\n🚀 PHASE 2: DATA SYNC & PRIVACY ASSIGNMENT\n{'='*60}")
    
    if not os.path.exists(input_file):
        return print(f"|-- [ERROR]: {input_file} missing.")

    # 1. Load the data
    gdf = gpd.read_parquet(input_file)

    # 2. THE PRO ADAPTATION: Only rename if the old name exists
    if 'area_ha' in gdf.columns:
        gdf = gdf.rename(columns={'area_ha': 'area_hectares'})
        print("|-- [LOG]: Adapted 'area_ha' to 'area_hectares' for SQL alignment.")
    elif 'area_hectares' in gdf.columns:
        print("|-- [LOG]: 'area_hectares' already aligned. No renaming needed.")
    else:
        # If neither exists, we have a data integrity problem!
        raise KeyError("❌ CRITICAL: No area column found in the Parquet file!")


    # ASSIGN OWNER:  
    gdf['owner_id'] = Config.DB_USER 

    # Filter out existing farms to maintain 'Idempotency'
    with engine.connect() as conn:
        existing_ids = pd.read_sql("SELECT farm_id FROM farm_boundaries", conn)['farm_id'].tolist()
    
    new_farms = gdf[~gdf['farm_id'].isin(existing_ids)]

    if new_farms.empty:
        print("|-- [LOG]: No new data to sync. Vault is up to date.")
        return

    spatial_dtypes = {
        "geom": Geometry("POLYGON", srid=4326),
        "geom_inner": Geometry("POLYGON", srid=4326)
    }

    new_farms.to_postgis(
        name="farm_boundaries",
        con=engine,
        if_exists="append", 
        index=False,
        dtype=spatial_dtypes
    )
    print(f"|-- [LOG]: ✅ {len(new_farms)} new farms secured with RLS.")



if __name__ == "__main__":
    ensure_vault_exists()
    sync_farms()
    print("\n🌟 DAY 3 COMPLETE: THE FORTRESS IS SECURED.")



🏗️  PHASE 1: SECURING THE SPATIAL VAULT (RLS & CONSTRAINTS)
|-- [LOG]: Vault structure, RLS policies, and Spatial Indexes are active.

🚀 PHASE 2: DATA SYNC & PRIVACY ASSIGNMENT
|-- [LOG]: Adapted 'area_ha' to 'area_hectares' for SQL alignment.
|-- [LOG]: No new data to sync. Vault is up to date.

🌟 DAY 3 COMPLETE: THE FORTRESS IS SECURED.
